In [ ]:
%load_ext autoreload
%autoreload 2

from LoRR_MAPF.envs.grid_world import LoRR
import planner_utils as planner
from tqdm import tqdm
import numpy as np

# Define a temporary map for testing
test_domain_path = "../example_problems/random.domain"
test_map_name = "random-32-32-20.map"
num_agents = 100
paths = [None,] * num_agents
room_paths = [None,] * num_agents

# Create test domain and map files
# Instantiate and test the environment
env = LoRR(domain_path=test_domain_path, num_agents=num_agents, map_name=test_map_name, render_mode="human", block_all_actions=False)
# env = LoRR(domain_path=test_domain_path, num_agents=500, map_name=test_map_name, render_mode="human")

obs, info = env.reset()  # Reset the environment
map_ = info['map']
ll_helpers = planner.load_low_level_helpers(test_map_name, map_)
hl_helpers = planner.load_high_level_helpers(test_map_name)

static_cost_map, mean_static_cost, map_dir, reflection_prob_map = ll_helpers
segmentation_map, connectivity_matrix, centroids = hl_helpers
print("initialized")

initialized


In [2]:
def get_current_agents_loc(obs: dict, map_: np.ndarray) -> np.ndarray:
    dir_agents_loc = np.zeros((4, map_.shape[0], map_.shape[1]))
    agents_loc, agents_dir = obs['agents']['pos'], obs['agents']['dir']
    for i in range(len(agents_loc)):
        agent_loc = agents_loc[i]
        agent_dir = agents_dir[i]
        dir_agents_loc[agent_dir, agent_loc[0], agent_loc[1]] = 1

    return dir_agents_loc

In [3]:
def follow_path_policy(paths, agents_pos, agents_dir):
    actions = [3,] * len(paths)
    for i, path in enumerate(paths):
        if not path:
            # Agent is at the goal
            continue
        if not (agents_pos[i][0], agents_pos[i][1]) == path[0][0]:
            actions[i] = 0 #Move forward
        elif path[0][1] != agents_dir[i]:
            incr = path[0][1]-agents_dir[i]
            if incr == 1 or incr == -3:
                actions[i] = 1
            elif incr == -1 or incr == 3:
                actions[i] = 2
    
    actions = [int(a) for a in actions]
    return actions

In [4]:
from multiprocess.shared_memory import SharedMemory

def create_shared_array(array):
    """Creates shared memory for a NumPy array and returns SharedMemory object + wrapped array."""
    shm = SharedMemory(create=True, size=array.nbytes)
    shared_array = np.ndarray(array.shape, dtype=array.dtype, buffer=shm.buf)
    shared_array[:] = array  # Copy data into shared memory
    return shm, shared_array

def initialize_shared_memory(map_, reflection_prob_map, segmentation_map, connectivity_matrix, centroids, static_cost_map):
    """Creates and returns shared memory objects for large numpy arrays."""
    shared_mem_objects = {}
    shared_arrays = {}

    large_arrays = {
        "map_": map_,
        "reflection_prob_map": reflection_prob_map,
        "segmentation_map": segmentation_map,
        "connectivity_matrix": connectivity_matrix,
        "centroids": centroids,
        "static_cost_map": static_cost_map
    }

    for name, array in large_arrays.items():
        shm, shared_arr = create_shared_array(array)
        shared_mem_objects[name] = shm
        shared_arrays[name] = shared_arr

    return shared_mem_objects, shared_arrays

In [5]:
shared_mem_objects, shared_arrays = initialize_shared_memory(map_, reflection_prob_map, segmentation_map, connectivity_matrix, centroids, static_cost_map)

In [ ]:
for i in tqdm(range(5000)):  # Run a few steps
    # if i % 100 == 0:
    #     obs, info = env.reset()
    actions = np.ones(num_agents) * 3 # Default to wait
    agents_loc = get_current_agents_loc(obs, map_)
    agents_pos = obs['agents']['pos']
    agents_dir = obs['agents']['dir']
    goals_pos = obs['target']


    paths, room_paths = planner.hierarchical_a_star_parallel(
        actions,
        agents_loc,
        agents_pos,
        agents_dir,
        goals_pos,
        paths,
        room_paths,
        shared_arrays,

        )

    actions = follow_path_policy(paths, agents_pos, agents_dir)
    obs, reward, terminated, truncated, info = env.step(actions)
    # env.render()

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 16/5000 [00:14<50:17,  1.65it/s] 

In [6]:
for i in tqdm(range(5000)):  # Run a few steps
    # if i % 100 == 0:
    #     obs, info = env.reset()
    actions = np.ones(num_agents) * 3 # Default to wait
    agents_loc = get_current_agents_loc(obs, map_)
    agents_pos = obs['agents']['pos']
    agents_dir = obs['agents']['dir']
    goals_pos = obs['target']


    paths, room_paths = planner.hierarchical_a_star(
        actions,
        agents_loc,
        agents_pos,
        agents_dir,
        goals_pos,
        map_,
        reflection_prob_map,
        segmentation_map,
        connectivity_matrix,
        centroids,
        static_cost_map,
        paths,
        room_paths,
        )

    actions = follow_path_policy(paths, agents_pos, agents_dir)
    obs, reward, terminated, truncated, info = env.step(actions)
    # env.render()

  0%|          | 0/5000 [00:00<?, ?it/s]/Users/alejandroaristizabal/Documents/LORR24_aristizabal95/python/planner_utils.py:238: NumbaTypeSafetyWarning: unsafe cast from int64 to uint8. Precision may be lost.
  curr = parent[curr]
  0%|          | 1/5000 [00:12<17:48:08, 12.82s/it]/Users/alejandroaristizabal/Documents/LORR24_aristizabal95/python/planner_utils.py:364: NumbaTypeSafetyWarning: unsafe cast from Tuple(UniTuple(int64 x 2), int64) to Tuple(UniTuple(int64 x 2), int8). Precision may be lost.
  curr = parent[curr]
 11%|█▏        | 568/5000 [02:05<16:18,  4.53it/s] 


KeyboardInterrupt: 

In [7]:
env.close()